In [1]:
import pandas as pd
import numpy as np
import gc
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, precision_recall_curve
import warnings

warnings.filterwarnings("ignore")

In [2]:
# ==========================================
# 1. CONFIGURATION & SETUP
# ==========================================
print("Initializing Environment...")
os.makedirs("models", exist_ok=True)
os.makedirs("curves", exist_ok=True)

PATH_17 = "CIC_IDS_2017.csv" 
PATH_18 = "CSC_CIC_2018Final.csv" 

experiments = {
    "DoS_Hulk": {"name": "DoS Hulk"},
    "Botnet": {"name": "Bot"},
    "Infiltration": {"name": "Infiltration"},
    "DoS_Slowloris": {"name": "DoS slowloris"},
    "DoS_GoldenEye": {"name": "DoS GoldenEye"},
    "DoS_Slowhttptest": {"name": "DoS Slowhttptest"},
    "FTP_BruteForce": {"name": "FTP-Patator"},
    "SSH_BruteForce": {"name": "SSH-Patator"},
    "Web_Brute_Force": {"name": "Web Attack - Brute Force"},
    "Web_XSS": {"name": "Web Attack - XSS"},
    "SQL_Injection": {"name": "Web Attack - Sql Injection"}
}

results_log = []

Initializing Environment...


In [3]:
# ==========================================
# 2. GLOBAL DATA LOADING & OPTIMIZATION
# ==========================================
print("\n" + "="*70)
print("LOADING MASSIVE DATASETS INTO MEMORY (SPEED-OPTIMIZED MODE)")
print("="*70)

# Using PyArrow engine for multi-threaded read speed
print("Reading CIC-17 CSV...")
df_global_17 = pd.read_csv(PATH_17, engine='pyarrow')

print("Reading CIC-18 CSV...")
df_global_18 = pd.read_csv(PATH_18, engine='pyarrow')

print("Downcasting float64 to float32 to double processing speed...")
# Identify and convert only numeric columns to prevent string corruption
float_cols_17 = df_global_17.select_dtypes(include=['float64']).columns
df_global_17[float_cols_17] = df_global_17[float_cols_17].astype(np.float32)

float_cols_18 = df_global_18.select_dtypes(include=['float64']).columns
df_global_18[float_cols_18] = df_global_18[float_cols_18].astype(np.float32)

print("Global data successfully loaded and optimized in RAM.")


LOADING MASSIVE DATASETS INTO MEMORY (SPEED-OPTIMIZED MODE)
Reading CIC-17 CSV...
Reading CIC-18 CSV...
Downcasting float64 to float32 to double processing speed...
Global data successfully loaded and optimized in RAM.


In [4]:
# ==========================================
# 3. CORE PIPELINE FUNCTIONS
# ==========================================

def get_balanced_subset(df, attack_name):
    """Filters attack rows + equal benign directly from an in-memory dataframe."""
    
    df_attack = df[df['label'] == attack_name]
    
    if len(df_attack) == 0:
        return None, None
        
    actual_size = len(df_attack)
    print(f"   -> Found {actual_size} '{attack_name}' rows. Extracting equal BENIGN rows.")
    
    df_benign = df[df['label'] == 'BENIGN'].sample(n=actual_size, random_state=42)
    
    df_subset = pd.concat([df_benign, df_attack]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_subset['label'] = df_subset['label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
    
    X = df_subset.drop(columns=['label'])
    y = df_subset['label']
    
    # We ONLY delete the temporary slices to free memory, NOT the global dataframe
    del df_attack, df_benign, df_subset
    gc.collect()
    return X, y

def train_and_apply_imputer(X_train, X_test, prefix_name):
    """Trains regression imputer with median fallback, applies to both sets, saves model."""
    imputer_models = {}
    all_features = X_train.columns
    COLS_WNA = X_train.columns[X_train.isna().any()].tolist()
    COLS_WONA = [c for c in all_features if c not in COLS_WNA]
    
    for target_col in COLS_WNA:
        train_data = X_train[X_train[target_col].notna()]
        if not train_data.empty:
            model = LinearRegression()
            model.fit(train_data[COLS_WONA], train_data[target_col])
            imputer_models[target_col] = model
            
            predict_data = X_train[X_train[target_col].isna()]
            if not predict_data.empty:
                X_train.loc[X_train[target_col].isna(), target_col] = model.predict(predict_data[COLS_WONA])
                
    fallback_medians = X_train.median().to_dict()
    joblib.dump({'regression_models': imputer_models, 'fallback_medians': fallback_medians, 'cols_wona': COLS_WONA}, f"models/Imputer_{prefix_name}.pkl")
    
    for col, model in imputer_models.items():
        if col in X_test.columns and X_test[col].isna().any():
            predict_data = X_test[X_test[col].isna()]
            if not predict_data.empty:
                X_test.loc[X_test[col].isna(), col] = model.predict(predict_data[COLS_WONA])
                
    for col in X_test.columns:
        if X_test[col].isna().any():
            X_test[col] = X_test[col].fillna(fallback_medians[col])
            
    return X_train, X_test

def save_evaluation_curves(y_true, y_probs, prefix_name):
    """Generates and saves ROC and PR curves directly to disk."""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc(fpr, tpr):.4f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.title(f'ROC Curve: {prefix_name}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    plt.subplot(1, 2, 2)
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    plt.plot(recall, precision, color='purple', lw=2)
    plt.title(f'PR Curve: {prefix_name}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"curves/Curves_{prefix_name}.png")
    plt.close()

In [5]:
# ==========================================
# 4. MASTER EXECUTION LOOP
# ==========================================

for exp_key, config in experiments.items():
    print(f"\n{'='*70}")
    print(f"STARTING CROSS-CORPUS EXPERIMENT: {exp_key}")
    print(f"{'='*70}")
    
    print(f"[{exp_key}] Extracting data from global CIC-17...")
    X_17, y_17 = get_balanced_subset(df_global_17, config["name"])
    
    print(f"[{exp_key}] Extracting data from global CIC-18...")
    X_18, y_18 = get_balanced_subset(df_global_18, config["name"])
    
    if X_17 is None or X_18 is None:
        print(f"--> SKIPPING {exp_key}: Attack not found in both datasets.")
        continue

    # ------------------------------------------
    # PHASE A: Train on CIC-17, Test on CIC-18
    # ------------------------------------------
    prefix_A = f"Train17_Test18_{exp_key}"
    print(f"\n---> [PHASE A] Running {prefix_A}...")
    
    X_test_18 = X_18[X_17.columns].copy()
    X_train_A, X_test_A = train_and_apply_imputer(X_17.copy(), X_test_18, prefix_A)
    
    scaler_A = MinMaxScaler()
    X_train_scaled_A = scaler_A.fit_transform(X_train_A).astype(np.float32)
    X_test_scaled_A = scaler_A.transform(X_test_A).astype(np.float32)
    joblib.dump(scaler_A, f"models/Scaler_{prefix_A}.pkl")
    
    rf_A = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
    rf_A.fit(X_train_scaled_A, y_17)
    joblib.dump(rf_A, f"models/RF_{prefix_A}.pkl")
    
    preds_A = rf_A.predict(X_test_scaled_A)
    probs_A = rf_A.predict_proba(X_test_scaled_A)[:, 1]
    save_evaluation_curves(y_18, probs_A, prefix_A)
    
    results_log.append({
        "Attack_Type": exp_key,
        "Train_Dataset": "CIC-17",
        "Test_Dataset": "CIC-18",
        "Benign_Precision": precision_score(y_18, preds_A, pos_label=0),
        "Benign_Recall": recall_score(y_18, preds_A, pos_label=0),
        "Benign_F1": f1_score(y_18, preds_A, pos_label=0),
        "Attack_Precision": precision_score(y_18, preds_A, pos_label=1),
        "Attack_Recall": recall_score(y_18, preds_A, pos_label=1),
        "Attack_F1": f1_score(y_18, preds_A, pos_label=1),
        "Macro_Avg_F1": f1_score(y_18, preds_A, average='macro')
    })
    
    # ------------------------------------------
    # PHASE B: Train on CIC-18, Test on CIC-17
    # ------------------------------------------
    prefix_B = f"Train18_Test17_{exp_key}"
    print(f"---> [PHASE B] Running {prefix_B}...")
    
    X_test_17 = X_17[X_18.columns].copy()
    X_train_B, X_test_B = train_and_apply_imputer(X_18.copy(), X_test_17, prefix_B)
    
    scaler_B = MinMaxScaler()
    X_train_scaled_B = scaler_B.fit_transform(X_train_B).astype(np.float32)
    X_test_scaled_B = scaler_B.transform(X_test_B).astype(np.float32)
    joblib.dump(scaler_B, f"models/Scaler_{prefix_B}.pkl")
    
    rf_B = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)
    rf_B.fit(X_train_scaled_B, y_18)
    joblib.dump(rf_B, f"models/RF_{prefix_B}.pkl")
    
    preds_B = rf_B.predict(X_test_scaled_B)
    probs_B = rf_B.predict_proba(X_test_scaled_B)[:, 1]
    save_evaluation_curves(y_17, probs_B, prefix_B)
    
    results_log.append({
        "Attack_Type": exp_key,
        "Train_Dataset": "CIC-18",
        "Test_Dataset": "CIC-17",
        "Benign_Precision": precision_score(y_17, preds_B, pos_label=0),
        "Benign_Recall": recall_score(y_17, preds_B, pos_label=0),
        "Benign_F1": f1_score(y_17, preds_B, pos_label=0),
        "Attack_Precision": precision_score(y_17, preds_B, pos_label=1),
        "Attack_Recall": recall_score(y_17, preds_B, pos_label=1),
        "Attack_F1": f1_score(y_17, preds_B, pos_label=1),
        "Macro_Avg_F1": f1_score(y_17, preds_B, average='macro')
    })

    print(f"Flushing iteration variables for {exp_key}...")
    del X_17, y_17, X_18, y_18, X_test_18, X_test_17
    del X_train_A, X_test_A, X_train_scaled_A, X_test_scaled_A, rf_A, preds_A, probs_A
    del X_train_B, X_test_B, X_train_scaled_B, X_test_scaled_B, rf_B, preds_B, probs_B
    gc.collect()


STARTING CROSS-CORPUS EXPERIMENT: DoS_Hulk
[DoS_Hulk] Extracting data from global CIC-17...
   -> Found 231072 'DoS Hulk' rows. Extracting equal BENIGN rows.
[DoS_Hulk] Extracting data from global CIC-18...
   -> Found 434873 'DoS Hulk' rows. Extracting equal BENIGN rows.

---> [PHASE A] Running Train17_Test18_DoS_Hulk...
---> [PHASE B] Running Train18_Test17_DoS_Hulk...
Flushing iteration variables for DoS_Hulk...

STARTING CROSS-CORPUS EXPERIMENT: Botnet
[Botnet] Extracting data from global CIC-17...
   -> Found 1966 'Bot' rows. Extracting equal BENIGN rows.
[Botnet] Extracting data from global CIC-18...
   -> Found 282310 'Bot' rows. Extracting equal BENIGN rows.

---> [PHASE A] Running Train17_Test18_Botnet...
---> [PHASE B] Running Train18_Test17_Botnet...
Flushing iteration variables for Botnet...

STARTING CROSS-CORPUS EXPERIMENT: Infiltration
[Infiltration] Extracting data from global CIC-17...
   -> Found 36 'Infiltration' rows. Extracting equal BENIGN rows.
[Infiltration] Ex

In [ ]:
# ==========================================
# 5. FINAL REPORT OUTPUT
# ==========================================
print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE. FINAL GENERALIZATION METRICS:")
print("="*80)

df_final_results = pd.DataFrame(results_log)
print(df_final_results.to_string(index=False))

df_final_results.to_csv("Final_Cross_Corpus_Results.csv", index=False)
print("\nSuccess! Full metrics table safely saved to 'Final_Cross_Corpus_Results.csv'.")


ALL EXPERIMENTS COMPLETE. FINAL GENERALIZATION METRICS:
     Attack_Type Train_Dataset Test_Dataset  Benign_Precision  Benign_Recall  Benign_F1  Attack_Precision  Attack_Recall  Attack_F1  Macro_Avg_F1
        DoS_Hulk        CIC-17       CIC-18          0.565211       0.972951   0.715038          0.902912       0.251556   0.393485      0.554262
        DoS_Hulk        CIC-18       CIC-17          0.571543       0.995452   0.726159          0.982392       0.253761   0.403336      0.564747
          Botnet        CIC-17       CIC-18          0.500304       0.999943   0.666924          0.957333       0.001272   0.002540      0.334732
          Botnet        CIC-18       CIC-17          0.522897       0.998983   0.686473          0.988636       0.088505   0.162465      0.424469
   DoS_Slowloris        CIC-17       CIC-18          0.989789       0.999028   0.994387          0.999019       0.989694   0.994334      0.994361
   DoS_Slowloris        CIC-18       CIC-17          0.641432      